# AntiAtropos SFT Training — Qwen3.5-4B with Unsloth QLoRA

Phase 1 of the training pipeline: teach the model strict SRE action grammar,
DAG physics intuition, and JSON-only output format via supervised fine-tuning.

**Model:** `Qwen/Qwen3.5-4B` (4-bit QLoRA via Unsloth)

**Target GPU:** T4 (14.56 GiB VRAM) — configs are flexible for A100 upgrade.

**Artifacts saved to:** `training/` folder

In [12]:
# ============================================================
# Cell 1: Install Dependencies
# IMPORTANT: After this cell finishes, RESTART THE RUNTIME
# (Runtime -> Restart session), then continue from Cell 2.
# ============================================================
!pip install -U transformers accelerate bitsandbytes
!pip install unsloth peft trl datasets
!pip install pydantic

print("\n" + "="*60)
print("INSTALLATION COMPLETE — PLEASE RESTART THE RUNTIME NOW")
print("Runtime -> Restart session, then run Cell 2.")
print("="*60)

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.4.8 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.6.2 which is incompatible.
unsloth-zoo 2026.4.9 requires transformers!=4.52.0,!=4.52.1,!

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.6.2
    Uninstalling transformers-5.6.2:
      Successfully uninstalled transformers-5.6.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.6.2
    Uninstalling transformers-5.6.2:
      Successfully uninstalled transformers-5.6.2



INSTALLATION COMPLETE — PLEASE RESTART THE RUNTIME NOW
Runtime -> Restart session, then run Cell 2.

INSTALLATION COMPLETE — PLEASE RESTART THE RUNTIME NOW
Runtime -> Restart session, then run Cell 2.


In [1]:
# ============================================================
# Cell 2: Verify Installation, Imports, Upload Project Files
# ============================================================
import importlib, sys, os, json, random, math, re, time
from pathlib import Path
from collections import Counter
from typing import Optional, List, Dict, Any

# Verify critical packages
import torch
import transformers
import peft
import trl

print(f"torch:         {torch.__version__}")
print(f"transformers:  {transformers.__version__}")
print(f"peft:          {peft.__version__}")
print(f"trl:           {trl.__version__}")
print(f"CUDA avail:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:           {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:    {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GiB")

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Paths
BASE_DIR = Path("/content/AntiAtropos")
OUTPUT_DIR = Path("/content/sft_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nOutput directory: {OUTPUT_DIR}")
print("Setup complete.")

torch:         2.10.0+cu128
transformers:  5.5.0
peft:          0.18.1
trl:           0.24.0
CUDA avail:    True
GPU:           Tesla T4


AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [2]:
# ============================================================
# Cell 3: HTTP Client for HF Space (replaces file upload)
# ============================================================
import requests

HF_SPACE_URL = "https://pranavkk-antiatropos.hf.space"
_session = requests.Session()

def env_reset(task_id="task-1", seed=None):
    payload = {"task_id": task_id}
    if seed is not None:
        payload["seed"] = seed
    resp = _session.post(f"{HF_SPACE_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, target_node_id, parameter):
    payload = {
        "action": {
            "action_type": action_type,
            "target_node_id": target_node_id,
            "parameter": parameter,
        }
    }
    resp = _session.post(f"{HF_SPACE_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Quick smoke test
test = env_reset("task-1")
print(f"Reset OK — task_id={test.get('observation', {}).get('task_id')}, step={test.get('observation', {}).get('step')}")

Reset OK — task_id=task-1, step=0


In [3]:
# ============================================================
# Cell 4: Verify HF Space connectivity + define helpers
# ============================================================
# No local imports needed — everything comes through the HTTP API.
# We still need these for the heuristic and data generation:
from enum import Enum
from pydantic import BaseModel, Field
import random, json, re, math
from collections import Counter
from pathlib import Path

# Re-define action types (no local models.py needed)
class ActionType(str, Enum):
    NO_OP = "NO_OP"
    SCALE_UP = "SCALE_UP"
    SCALE_DOWN = "SCALE_DOWN"
    REROUTE_TRAFFIC = "REROUTE_TRAFFIC"
    SHED_LOAD = "SHED_LOAD"

VALID_ACTIONS = [a.value for a in ActionType]
VALID_NODES = ["node-0", "node-1", "node-2", "node-3", "node-4"]

# Paths
OUTPUT_DIR = Path("/content/sft_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

# Test step
step_result = env_step("NO_OP", "node-0", 0.0)
print(f"Step OK — reward={step_result.get('reward')}, done={step_result.get('done')}")
print("HF Space connectivity verified.")

Step OK — reward=0.6147159298543577, done=False
HF Space connectivity verified.


In [4]:
# ============================================================
# Cell 5: Load Qwen3.5-4B with Unsloth (4-bit QLoRA)
# ============================================================
from unsloth import FastLanguageModel

# ---- Config struct (easy to change for A100) ----
class ModelConfig:
    model_name: str = "Qwen/Qwen3.5-4B"
    max_seq_length: int = 2048      # T4 safe; A100: 4096
    lora_rank: int = 16             # T4 safe; A100: 32
    lora_alpha: int = 16
    load_in_4bit: bool = True       # Keep True even on A100 for QLoRA
    dtype = None                    # Auto-detect (bf16 on T4+)

cfg = ModelConfig()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg.model_name,
    max_seq_length=cfg.max_seq_length,
    load_in_4bit=cfg.load_in_4bit,
    dtype=cfg.dtype,
    trust_remote_code=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Long context autograd
    random_state=SEED,
)

# Verify VRAM
if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM used: {vram_used:.2f} GiB / {vram_total:.2f} GiB")
    print(f"VRAM free: {vram_total - vram_used:.2f} GiB")

print(f"\nModel: {cfg.model_name}")
print(f"LoRA rank: {cfg.lora_rank}, alpha: {cfg.lora_alpha}")
print(f"Max seq length: {cfg.max_seq_length}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print("Model loaded successfully.")

/tmp/ipykernel_6872/1894257659.py:4: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.8: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [5]:
# ============================================================
# Cell 6: SFT Data Generation — Heuristic + Expert Dataset
# Generates a comprehensive, balanced training dataset covering
# all 5 action types, all 5 nodes, DAG physics, and edge cases.
# ============================================================

VALID_ACTIONS = ["NO_OP", "SCALE_UP", "SCALE_DOWN", "REROUTE_TRAFFIC", "SHED_LOAD"]
VALID_NODES = ["node-0", "node-1", "node-2", "node-3", "node-4"]
MAX_QUEUE_NORM = 200.0
MAX_LATENCY_NORM = 1000.0
MAX_REQUEST_RATE_NORM = 100.0

# ---- System prompt (shared across all examples) ----
SYSTEM_PROMPT = """You are an autonomous SRE controller managing a five-node microservice cluster.

CLUSTER TOPOLOGY (traffic flows parent -> children):
  node-0 (VIP payment gateway) -> node-1, node-2
  node-2 (catalog) -> node-3 (inventory)
  node-4 (auth, independent ingress)
FAILED nodes have outflow=0 — their children are starved.
Backpressure: overloaded children reduce parent capacity.

ACTIONS (new capacity takes 5 ticks to boot):
  SCALE_UP <node> <amount>   — add capacity (0.3-0.5 normal, 0.6-0.8 heavy surge)
  SCALE_DOWN <node> <amount>  — remove capacity (0.2-0.4 safe, 0.5-0.7 aggressive)
  REROUTE_TRAFFIC <node> <fraction> — move traffic AWAY from this node to healthy peers (0.3-0.7)
  SHED_LOAD <node> <fraction>  — drop incoming traffic (0.3-0.5), NEVER on node-0 (VIP)
  NO_OP — do nothing when cluster is healthy

CRITICAL RULES:
  - node-0 is the VIP payment gateway — NEVER shed its traffic
  - REROUTE_TRAFFIC moves traffic AWAY FROM the target node
  - SCALE_UP clears DEGRADED status on the target node
  - Boot delay is 5 ticks — plan ahead for scaling
  - Use English for reasoning, JSON for the action

REWARD PRIORITIES (in order):
  1. Avoid SLA violations (latency > 200ms or error rate > 5%)
  2. Keep queues low (growing queues = destabilizing system)
  3. Don't over-provision (excess capacity costs money)

You MUST respond with one sentence of English reasoning, then a JSON action.
The JSON must use EXACTLY these keys: action_type, target_node_id, parameter.
action_type must be one of: SCALE_UP, SCALE_DOWN, REROUTE_TRAFFIC, SHED_LOAD, NO_OP.
target_node_id must be one of: node-0, node-1, node-2, node-3, node-4.
parameter must be a float between 0.0 and 10.0."""


def format_observation(obs_dict, task_id, step, max_steps, reward=0.0, sla_violations=0):
    """Convert API observation dict to natural-language string."""
    lines = [f"Task: {task_id}  Step: {step}/{max_steps}  Reward: {reward:.3f}  SLA violations: {sla_violations}"]
    lines.append("")
    lines.append("Node states:")
    for n in obs_dict.get("nodes", []):
        vip = " (VIP)" if n.get("is_vip") else ""
        status = n.get("status", "HEALTHY")
        # API returns normalized [0,1]; denormalize for readability
        q = n.get("queue_depth", 0) * 200
        cap = n.get("capacity", 0) * 5
        pending = n.get("pending_capacity", 0) * 5
        inc = n.get("incoming_request_rate", 0) * 100
        lat = n.get("latency_ms", 0) * 1000
        outflow = n.get("outflow_rate", 0) * 100
        failed = " [FAILED, outflow=0]" if status == "FAILED" else ""
        degraded = " [DEGRADED]" if status == "DEGRADED" else ""
        pending_str = f" (+{pending:.0f} booting)" if pending > 0 else ""
        lines.append(
            f"  {n['node_id']}{vip}: queue={int(q)}, capacity={cap:.0f}{pending_str}, "
            f"incoming={inc:.0f}, latency={lat:.0f}ms, outflow={outflow:.0f}{failed}{degraded}"
        )
    return "\n".join(lines)


def heuristic_action(obs_dict, task_id):
    """Enhanced heuristic that covers all action types. Works with normalized API data."""
    nodes = obs_dict.get("nodes", [])

    # Denormalize for threshold comparisons
    total_queue = sum(n["queue_depth"] * 200 for n in nodes)
    avg_latency = sum(n["latency_ms"] for n in nodes) / len(nodes) if nodes else 0  # already [0,1]
    failed_nodes = [n for n in nodes if n.get("status") == "FAILED"]
    degraded_nodes = [n for n in nodes if n.get("status") == "DEGRADED"]

    # PRIORITY 1: REROUTE away from FAILED nodes
    if failed_nodes:
        target = failed_nodes[0]
        return ActionType.REROUTE_TRAFFIC, target["node_id"], 0.7

    # PRIORITY 2: SHED_LOAD on non-critical overloaded nodes
    non_critical_overloaded = [
        n for n in nodes
        if n["queue_depth"] > 0.6 and n["node_id"] != "node-0"
        and n.get("status") != "FAILED"
    ]
    if non_critical_overloaded and avg_latency > 0.05:
        shed_candidates = [n for n in non_critical_overloaded if n["node_id"] in ["node-3", "node-4"]]
        target = shed_candidates[0] if shed_candidates else non_critical_overloaded[0]
        return ActionType.SHED_LOAD, target["node_id"], 0.4

    # PRIORITY 3: SCALE_UP stressed nodes
    if avg_latency > 0.03 or total_queue > 200:
        target = max(nodes, key=lambda n: n["queue_depth"])
        param = 0.6 if target["queue_depth"] > 0.75 else 0.4
        return ActionType.SCALE_UP, target["node_id"], param

    # PRIORITY 4: SCALE_DOWN idle/overprovisioned nodes
    non_vips = [n for n in nodes if not n.get("is_vip", False) and n.get("status") != "FAILED"]
    if non_vips and avg_latency < 0.025 and total_queue < 50:
        overprov = [n for n in non_vips if n.get("capacity", 0) > 0.6]
        if overprov:
            target = max(overprov, key=lambda n: n.get("capacity", 0))
            return ActionType.SCALE_DOWN, target["node_id"], 0.3

    # PRIORITY 5: NO_OP when healthy
    target = max(nodes, key=lambda n: n["queue_depth"])
    return ActionType.NO_OP, target["node_id"], 0.0


def generate_reasoning(action_type, target_node_id, parameter, obs_dict, task_id):
    """Generate an English reasoning sentence for the chosen action."""
    nodes = obs_dict.get("nodes", [])
    node_map = {n["node_id"]: n for n in nodes}
    target = node_map.get(target_node_id, {})
    q = target.get("queue_depth", 0) * 200

    if action_type == ActionType.SCALE_UP:
        if q > 150:
            return f"Node {target_node_id} is near FATAL with queue={int(q)}, scaling up immediately to prevent cascading failure."
        elif q > 80:
            return f"Node {target_node_id} is DEGRADED with queue={int(q)}, scaling up to restore service capacity."
        else:
            return f"Node {target_node_id} queue={int(q)} is rising, proactively scaling up to stay ahead of demand."
    elif action_type == ActionType.SCALE_DOWN:
        return f"Cluster is stable with low queues, scaling down {target_node_id} to reduce infrastructure cost."
    elif action_type == ActionType.REROUTE_TRAFFIC:
        status = target.get("status", "HEALTHY")
        if status == "FAILED":
            return f"Node {target_node_id} has FAILED with outflow=0, rerouting its traffic to healthy peers to prevent request loss."
        else:
            return f"Node {target_node_id} is overloaded with queue={int(q)}, rerouting traffic to healthy peers to reduce pressure."
    elif action_type == ActionType.SHED_LOAD:
        return f"Node {target_node_id} is overloaded but non-critical, shedding {parameter:.0%} of incoming traffic to protect cluster stability."
    elif action_type == ActionType.NO_OP:
        return f"All nodes are healthy with low queues, no intervention needed."
    return "Taking action based on current cluster state."


# ---- Expert augmentation examples ----
EXPERT_EXAMPLES = []

# Expert: REROUTE_TRAFFIC on FAILED node (task-2 scenario)
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-2  Step: 25/60  Reward: 0.320  SLA violations: 3

Node states:
  node-0 (VIP): queue=55, capacity=3, incoming=47, latency=25ms, outflow=45
  node-1: queue=0, capacity=0, incoming=23, latency=inf, outflow=0 [FAILED, outflow=0]
  node-2: queue=18, capacity=3, incoming=23, latency=22ms, outflow=23
  node-3: queue=2, capacity=3, incoming=12, latency=20ms, outflow=12
  node-4: queue=12, capacity=3, incoming=47, latency=21ms, outflow=45""",
    "assistant": "Node node-1 has FAILED with outflow=0, rerouting its traffic to healthy peers to prevent request loss. {\"action_type\": \"REROUTE_TRAFFIC\", \"target_node_id\": \"node-1\", \"parameter\": 0.7}",
})

EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-2  Step: 30/60  Reward: 0.280  SLA violations: 5

Node states:
  node-0 (VIP): queue=65, capacity=3, incoming=47, latency=28ms, outflow=43
  node-1: queue=0, capacity=0, incoming=24, latency=inf, outflow=0 [FAILED, outflow=0]
  node-2: queue=90, capacity=3, incoming=24, latency=35ms, outflow=24
  node-3: queue=5, capacity=3, incoming=12, latency=21ms, outflow=12
  node-4: queue=15, capacity=3, incoming=47, latency=22ms, outflow=45""",
    "assistant": "Node node-1 is FAILED and node-0 is still sending 50% traffic to it, rerouting away from node-1 to save that wasted traffic. {\"action_type\": \"REROUTE_TRAFFIC\", \"target_node_id\": \"node-1\", \"parameter\": 0.8}",
})

# Expert: SHED_LOAD on non-critical node (task-3 surge)
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-3  Step: 35/60  Reward: 0.250  SLA violations: 4

Node states:
  node-0 (VIP): queue=10, capacity=3, incoming=30, latency=20ms, outflow=30
  node-1: queue=45, capacity=3, incoming=75, latency=22ms, outflow=45
  node-2: queue=160, capacity=3, incoming=75, latency=55ms, outflow=45
  node-3: queue=140, capacity=3, incoming=45, latency=48ms, outflow=45
  node-4: queue=5, capacity=3, incoming=30, latency=20ms, outflow=30""",
    "assistant": "Node node-3 (inventory) is non-critical and overloaded at queue=140, shedding 40% of its traffic to prevent cascade while scaling handles the surge upstream. {\"action_type\": \"SHED_LOAD\", \"target_node_id\": \"node-3\", \"parameter\": 0.4}",
})

EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-3  Step: 38/60  Reward: 0.220  SLA violations: 6

Node states:
  node-0 (VIP): queue=8, capacity=3, incoming=30, latency=20ms, outflow=30
  node-1: queue=50, capacity=5, incoming=80, latency=21ms, outflow=75
  node-2: queue=170, capacity=4, incoming=80, latency=62ms, outflow=60
  node-3: queue=155, capacity=3, incoming=60, latency=52ms, outflow=45
  node-4: queue=130, capacity=3, incoming=75, latency=35ms, outflow=45""",
    "assistant": "Node node-4 (auth) is independent and overloaded at queue=130, shedding 35% of its incoming traffic since it is not on the critical DAG path. {\"action_type\": \"SHED_LOAD\", \"target_node_id\": \"node-4\", \"parameter\": 0.35}",
})

# Expert: SCALE_DOWN on idle node
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 80/100  Reward: 0.650  SLA violations: 0

Node states:
  node-0 (VIP): queue=5, capacity=5, incoming=35, latency=20ms, outflow=35
  node-1: queue=3, capacity=4, incoming=18, latency=20ms, outflow=18
  node-2: queue=4, capacity=5, incoming=18, latency=20ms, outflow=18
  node-3: queue=2, capacity=4, incoming=9, latency=20ms, outflow=9
  node-4: queue=3, capacity=6, incoming=35, latency=20ms, outflow=35""",
    "assistant": "Cluster is very stable with all queues under 10, scaling down node-4 which has 6 capacity units but only needs 3 for its traffic. {\"action_type\": \"SCALE_DOWN\", \"target_node_id\": \"node-4\", \"parameter\": 0.4}",
})

# Expert: DAG bottleneck — scale downstream, not ingress
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 45/100  Reward: 0.350  SLA violations: 2

Node states:
  node-0 (VIP): queue=30, capacity=3, incoming=44, latency=21ms, outflow=42
  node-1: queue=8, capacity=3, incoming=22, latency=20ms, outflow=22
  node-2: queue=185, capacity=3, incoming=22, latency=75ms, outflow=22
  node-3: queue=170, capacity=3, incoming=22, latency=65ms, outflow=22
  node-4: queue=10, capacity=3, incoming=43, latency=20ms, outflow=43""",
    "assistant": "The bottleneck is downstream at node-2 (queue=185) and node-3 (queue=170), not at ingress node-0. Backpressure from node-2/3 is throttling node-0. Scaling up node-2 to break the backpressure chain. {\"action_type\": \"SCALE_UP\", \"target_node_id\": \"node-2\", \"parameter\": 0.6}",
})

# Expert: NO_OP on healthy cluster
EXPERT_EXAMPLES.append({
    "system": SYSTEM_PROMPT,
    "user": """Task: task-1  Step: 10/100  Reward: 0.700  SLA violations: 0

Node states:
  node-0 (VIP): queue=8, capacity=3, incoming=42, latency=20ms, outflow=42
  node-1: queue=5, capacity=3, incoming=21, latency=20ms, outflow=21
  node-2: queue=6, capacity=3, incoming=21, latency=20ms, outflow=21
  node-3: queue=3, capacity=3, incoming=10, latency=20ms, outflow=10
  node-4: queue=7, capacity=3, incoming=42, latency=20ms, outflow=42""",
    "assistant": "All nodes are healthy with low queues and normal latency, no intervention needed. {\"action_type\": \"NO_OP\", \"target_node_id\": \"node-0\", \"parameter\": 0.0}",
})


print("Heuristic and expert examples defined.")
print(f"Expert examples: {len(EXPERT_EXAMPLES)}")
print(f"Action types in experts: {set('REROUTE_TRAFFIC' if 'REROUTE' in e['assistant'] else 'SHED_LOAD' if 'SHED' in e['assistant'] else 'SCALE_DOWN' if 'SCALE_DOWN' in e['assistant'] else 'SCALE_UP' if 'SCALE_UP' in e['assistant'] else 'NO_OP' for e in EXPERT_EXAMPLES)}")

Heuristic and expert examples defined.
Expert examples: 7
Action types in experts: {'SCALE_UP', 'NO_OP', 'SCALE_DOWN', 'REROUTE_TRAFFIC', 'SHED_LOAD'}


In [ ]:
# ============================================================
# Cell 7: Run Episodes & Build SFT Dataset
# ============================================================

NUM_EPISODES_PER_TASK = 20
MAX_STEPS = 60
NO_OP_CAP = 0.40

all_examples = []
action_counts = Counter()
node_counts = Counter()
reward_sum = 0.0
reward_count = 0

for task_id in ["task-1", "task-2", "task-3"]:
    print(f"\n--- Generating episodes for {task_id} ---")
    for ep_idx in range(NUM_EPISODES_PER_TASK):
        seed = SEED + ep_idx * 100 + hash(task_id) % 1000

        # Reset via HF Space
        reset_resp = env_reset(task_id=task_id, seed=seed)
        obs_dict = reset_resp.get("observation", reset_resp)
        sla_violations = obs_dict.get("sla_violations", 0)
        episode_reward = 0.0

        for step in range(1, MAX_STEPS + 1):
            obs_text = format_observation(obs_dict, task_id, step, MAX_STEPS, episode_reward, sla_violations)
            action_type, target_node_id, parameter = heuristic_action(obs_dict, task_id)
            reasoning = generate_reasoning(action_type, target_node_id, parameter, obs_dict, task_id)

            action_json = json.dumps({
                "action_type": action_type.value,
                "target_node_id": target_node_id,
                "parameter": round(float(parameter), 4),
            })
            assistant_text = f"{reasoning} {action_json}"

            all_examples.append({
                "system": SYSTEM_PROMPT,
                "user": obs_text,
                "assistant": assistant_text,
                "task_id": task_id,
                "action_type": action_type.value,
                "target_node_id": target_node_id,
                "reward": episode_reward,
            })

            action_counts[action_type.value] += 1
            node_counts[target_node_id] += 1

            # Step via HF Space
            step_resp = env_step(action_type.value, target_node_id, parameter)
            obs_dict = step_resp.get("observation", step_resp)
            episode_reward = step_resp.get("reward", episode_reward)
            reward_sum += episode_reward
            reward_count += 1
            sla_violations = obs_dict.get("sla_violations", sla_violations)

            if step_resp.get("done", False):
                break

        if (ep_idx + 1) % 5 == 0:
            print(f"  Episode {ep_idx+1}/{NUM_EPISODES_PER_TASK} done")

print(f"\nHeuristic episodes generated: {len(all_examples)} examples")
# ---- Add expert examples ----
for ex in EXPERT_EXAMPLES:
    # Parse action type from the assistant text
    at_match = re.search(r'"action_type":\s*"(\w+)"', ex["assistant"])
    tn_match = re.search(r'"target_node_id":\s*"(node-\d+)"', ex["assistant"])
    if at_match and tn_match:
        all_examples.append({
            "system": ex["system"],
            "user": ex["user"],
            "assistant": ex["assistant"],
            "task_id": "expert",
            "action_type": at_match.group(1),
            "target_node_id": tn_match.group(1),
            "reward": 0.0,
        })
        action_counts[at_match.group(1)] += 1
        node_counts[tn_match.group(1)] += 1

print(f"After expert augmentation: {len(all_examples)} examples")

# ---- Filter: cap NO_OP at 40% ----
noop_examples = [e for e in all_examples if e["action_type"] == "NO_OP"]
non_noop_examples = [e for e in all_examples if e["action_type"] != "NO_OP"]
max_noop = int(len(all_examples) * NO_OP_CAP)
if len(noop_examples) > max_noop:
    random.shuffle(noop_examples)
    noop_examples = noop_examples[:max_noop]
    all_examples = non_noop_examples + noop_examples
    print(f"NO_OP capped: keeping {max_noop} of {len(noop_examples)} NO_OP examples")

# ---- Deduplicate ----
seen = set()
unique_examples = []
for e in all_examples:
    key = (e["user"][:100], e["action_type"], e["target_node_id"])
    if key not in seen:
        seen.add(key)
        unique_examples.append(e)
all_examples = unique_examples
print(f"After dedup: {len(all_examples)} examples")

# ---- Shuffle ----
random.shuffle(all_examples)

# ---- Train/Test/Val split (70/15/15, stratified by action_type) ----
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_ex, temp_ex = train_test_split(
    all_examples, test_size=0.30, random_state=SEED,
    stratify=[e["action_type"] for e in all_examples],
)
# Second split: 15% test, 15% val from temp
test_ex, val_ex = train_test_split(
    temp_ex, test_size=0.50, random_state=SEED,
    stratify=[e["action_type"] for e in temp_ex],
)

print(f"\nDataset split:")
print(f"  Train: {len(train_ex)}")
print(f"  Test:  {len(test_ex)}")
print(f"  Val:   {len(val_ex)}")

# ---- Save as JSONL ----
def save_jsonl(examples, path):
    with open(path, "w") as f:
        for ex in examples:
            # Save in chat format for SFTTrainer
            record = {
                "messages": [
                    {"role": "system", "content": ex["system"]},
                    {"role": "user", "content": ex["user"]},
                    {"role": "assistant", "content": ex["assistant"]},
                ],
                "task_id": ex.get("task_id", ""),
                "action_type": ex.get("action_type", ""),
                "target_node_id": ex.get("target_node_id", ""),
            }
            f.write(json.dumps(record) + "\n")

save_jsonl(train_ex, OUTPUT_DIR / "sft_train.jsonl")
save_jsonl(test_ex, OUTPUT_DIR / "sft_test.jsonl")
save_jsonl(val_ex, OUTPUT_DIR / "sft_val.jsonl")

# ---- Print statistics ----
final_action_counts = Counter(e["action_type"] for e in all_examples)
final_node_counts = Counter(e["target_node_id"] for e in all_examples)
total = len(all_examples)

print(f"\n{'='*60}")
print(f"DATASET STATISTICS ({total} total examples)")
print(f"{'='*60}")
print(f"\nAction type distribution:")
for at in VALID_ACTIONS:
    cnt = final_action_counts.get(at, 0)
    pct = 100 * cnt / total if total > 0 else 0
    flag = " <-- UNDERREPRESENTED" if pct < 3 and at != "NO_OP" else ""
    print(f"  {at:20s}: {cnt:5d} ({pct:5.1f}%){flag}")

print(f"\nTarget node distribution:")
for nid in VALID_NODES:
    cnt = final_node_counts.get(nid, 0)
    pct = 100 * cnt / total if total > 0 else 0
    flag = " <-- UNDERREPRESENTED" if pct < 5 else ""
    print(f"  {nid:10s}: {cnt:5d} ({pct:5.1f}%){flag}")

print(f"\nAverage reward: {reward_sum / reward_count:.4f}" if reward_count > 0 else "\nNo rewards computed")
print(f"\nFiles saved to {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.glob("sft_*.jsonl")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")


--- Generating episodes for task-1 ---
  Episode 5/20 done
  Episode 5/20 done


In [ ]:
# ============================================================
# Cell 8: SFT Training with Unsloth + SFTTrainer (QLoRA)
# ============================================================
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# ---- Load dataset ----
def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_jsonl(OUTPUT_DIR / "sft_train.jsonl")
val_records = load_jsonl(OUTPUT_DIR / "sft_val.jsonl")

train_dataset = Dataset.from_list(train_records)
val_dataset = Dataset.from_list(val_records)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"Sample record keys: {train_dataset[0].keys()}")

# ---- Training config (T4-safe, A100-flexible) ----
class TrainConfig:
    # T4 defaults
    per_device_train_batch_size: int = 1       # T4; A100: 4
    gradient_accumulation_steps: int = 8       # T4; A100: 4
    learning_rate: float = 2e-4
    num_train_epochs: int = 2                  # T4; A100: 3
    max_seq_length: int = 2048                 # Must match model config
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    logging_steps: int = 10
    save_steps: int = 100
    eval_strategy: str = "steps"
    eval_steps: int = 100
    save_total_limit: int = 3
    bf16: bool = True
    optim: str = "adamw_8bit"
    output_dir: str = str(OUTPUT_DIR / "sft_lora_adapter")

tc = TrainConfig()

# ---- Set up trainer ----
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=tc.per_device_train_batch_size,
        gradient_accumulation_steps=tc.gradient_accumulation_steps,
        learning_rate=tc.learning_rate,
        num_train_epochs=tc.num_train_epochs,
        max_seq_length=tc.max_seq_length,
        warmup_ratio=tc.warmup_ratio,
        weight_decay=tc.weight_decay,
        logging_steps=tc.logging_steps,
        save_steps=tc.save_steps,
        eval_strategy=tc.eval_strategy,
        eval_steps=tc.eval_steps,
        save_total_limit=tc.save_total_limit,
        bf16=tc.bf16,
        optim=tc.optim,
        output_dir=tc.output_dir,
        dataset_text_field="",
        packing=False,
    ),
    formatting_func=lambda example: tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    ),
)

print("\nTrainer configured.")
print(f"Effective batch size: {tc.per_device_train_batch_size * tc.gradient_accumulation_steps}")
print(f"Total epochs: {tc.num_train_epochs}")
print(f"Learning rate: {tc.learning_rate}")
print(f"Optimizer: {tc.optim}")
print(f"Output dir: {tc.output_dir}")
print("\nStarting training...")

# ---- Train! ----
train_result = trainer.train()

# ---- Print results ----
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Train loss: {train_result.training_loss:.4f}")
print(f"Train runtime: {train_result.metrics['train_runtime']:.1f}s")

# ---- Save final adapter ----
trainer.save_model(tc.output_dir)
tokenizer.save_pretrained(tc.output_dir)
print(f"\nLoRA adapter saved to: {tc.output_dir}")

# ---- Eval ----
eval_result = trainer.evaluate()
print(f"\nEval loss: {eval_result.get('eval_loss', 'N/A')}")
if 'eval_loss' in eval_result:
    print(f"Train/Eval loss ratio: {train_result.training_loss / eval_result['eval_loss']:.3f}")

In [ ]:
# ============================================================
# Cell 9: Lightweight Quality Gate — Live Simulated Episodes
# Compare SFT model vs heuristic baseline on simulated episodes.
# ============================================================
from peft import PeftModel

# ---- Load the trained SFT adapter ----
print("Loading SFT adapter for evaluation...")
FastLanguageModel.for_inference(model)  # Enable inference mode
print("Model ready for inference.")


def parse_model_action(text):
    """Extract SREAction from model output text."""
    try:
        # Find JSON object
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end < start:
            return None, "no JSON found"
        obj = json.loads(text[start:end+1])
        at = str(obj.get("action_type", "")).upper()
        if at not in VALID_ACTIONS:
            return None, f"invalid action_type: {at}"
        nid = str(obj.get("target_node_id", ""))
        if nid not in VALID_NODES:
            return None, f"invalid target_node_id: {nid}"
        param = float(obj.get("parameter", 0.0))
        return SREAction(
            action_type=ActionType(at),
            target_node_id=nid,
            parameter=param,
        ), "ok"
    except Exception as e:
        return None, str(e)


def run_episode_with_model(sim, task_id, max_steps, model, tokenizer):
    """Run one episode using the SFT model."""
    sim.reset(task_id=task_id, seed=None)
    prev_lyapunov = compute_lyapunov_graph(sim.state(for_agent=False), CLUSTER_TOPOLOGY)
    rewards = []
    observations = []
    invalid_actions = 0
    crashes = 0
    action_log = []

    for step in range(1, max_steps + 1):
        obs_text = format_observation(sim, task_id, step, max_steps,
                                       rewards[-1] if rewards else 0.0,
                                       sum(1 for r in rewards if r < 0.3))
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs_text},
        ]
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                temperature=1.0,
            )
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        )

        action, err = parse_model_action(generated)
        if action is None:
            crashes += 1
            # Fallback to NO_OP so the episode can continue
            action = SREAction(action_type=ActionType.NO_OP, target_node_id="node-0", parameter=0.0)
            action_log.append(f"step={step} INVALID ({err})")
        else:
            action_log.append(f"step={step} {action.action_type.value} {action.target_node_id} p={action.parameter:.2f}")

        # Step the simulator
        had_effect = sim.apply_action(action)
        if not had_effect:
            invalid_actions += 1
        sim.tick()

        # Compute reward
        nodes_state = sim.state(for_agent=False)
        curr_lyapunov = compute_lyapunov_graph(nodes_state, CLUSTER_TOPOLOGY)
        barrier = compute_barrier(nodes_state)
        raw_reward = compute_reward(
            v_prev=prev_lyapunov, v_curr=curr_lyapunov,
            cost=0.0, sla_violation_step=0.0, barrier=barrier,
        )
        norm_reward = normalize_reward(raw_reward)
        rewards.append(norm_reward)
        prev_lyapunov = curr_lyapunov

    return {
        "rewards": rewards,
        "avg_reward": sum(rewards) / len(rewards) if rewards else 0.0,
        "crashes": crashes,
        "invalid_actions": invalid_actions,
        "action_log": action_log,
    }


def run_episode_with_heuristic(sim, task_id, max_steps):
    """Run one episode using the heuristic baseline."""
    sim.reset(task_id=task_id, seed=None)
    prev_lyapunov = compute_lyapunov_graph(sim.state(for_agent=False), CLUSTER_TOPOLOGY)
    rewards = []

    for step in range(1, max_steps + 1):
        action = heuristic_action(sim, task_id)
        sim.apply_action(action)
        sim.tick()

        nodes_state = sim.state(for_agent=False)
        curr_lyapunov = compute_lyapunov_graph(nodes_state, CLUSTER_TOPOLOGY)
        barrier = compute_barrier(nodes_state)
        raw_reward = compute_reward(
            v_prev=prev_lyapunov, v_curr=curr_lyapunov,
            cost=0.0, sla_violation_step=0.0, barrier=barrier,
        )
        norm_reward = normalize_reward(raw_reward)
        rewards.append(norm_reward)
        prev_lyapunov = curr_lyapunov

    return {
        "rewards": rewards,
        "avg_reward": sum(rewards) / len(rewards) if rewards else 0.0,
    }


# ---- Run evaluation ----
EVAL_EPISODES = 3
EVAL_STEPS = 60

print(f"Running {EVAL_EPISODES} episodes per task ({EVAL_STEPS} steps each)...")
print(f"{'='*70}")

results = {}
for task_id in ["task-1", "task-2", "task-3"]:
    sft_rewards = []
    heuristic_rewards = []
    total_crashes = 0
    total_invalid = 0

    for ep in range(EVAL_EPISODES):
        sim = ClusterSimulator(n_nodes=5, task_id=task_id)

        # SFT model episode
        sft_result = run_episode_with_model(sim, task_id, EVAL_STEPS, model, tokenizer)
        sft_rewards.append(sft_result["avg_reward"])
        total_crashes += sft_result["crashes"]
        total_invalid += sft_result["invalid_actions"]

        # Heuristic episode
        sim2 = ClusterSimulator(n_nodes=5, task_id=task_id)
        heur_result = run_episode_with_heuristic(sim2, task_id, EVAL_STEPS)
        heuristic_rewards.append(heur_result["avg_reward"])

    sft_avg = sum(sft_rewards) / len(sft_rewards)
    heur_avg = sum(heuristic_rewards) / len(heuristic_rewards)

    results[task_id] = {
        "sft_avg_reward": sft_avg,
        "heuristic_avg_reward": heur_avg,
        "crashes": total_crashes,
        "invalid_actions": total_invalid,
        "sft_beats_heuristic": sft_avg >= heur_avg,
    }

    status = "SFT WINS" if sft_avg >= heur_avg else "HEURISTIC WINS"
    print(f"\n{task_id}:")
    print(f"  SFT avg reward:       {sft_avg:.4f}")
    print(f"  Heuristic avg reward: {heur_avg:.4f}")
    print(f"  Result: {status}")
    print(f"  Crashes: {total_crashes}, Invalid actions: {total_invalid}")

# ---- Summary ----
tasks_won = sum(1 for r in results.values() if r["sft_beats_heuristic"])
total_crashes = sum(r["crashes"] for r in results.values())

print(f"\n{'='*70}")
print(f"QUALITY GATE SUMMARY")
print(f"{'='*70}")
print(f"SFT beats heuristic on: {tasks_won}/3 tasks")
print(f"Total crashes: {total_crashes}")
print(f"\nGate results:")
gate_pass = total_crashes == 0 and tasks_won >= 2
print(f"  Zero crashes: {'PASS' if total_crashes == 0 else 'FAIL'}")
print(f"  Beat heuristic >= 2/3: {'PASS' if tasks_won >= 2 else 'FAIL'}")
print(f"\nOverall: {'GATE PASSED' if gate_pass else 'GATE FAILED — consider more SFT epochs or larger dataset'}")

## Summary & Next Steps

**Artifacts saved to `training/`:**
- `sft_train.jsonl` / `sft_test.jsonl` / `sft_val.jsonl` — Training dataset
- `sft_lora_adapter/` — LoRA adapter weights

**What was validated:**
- Model produces valid JSON SRE actions
- Model doesn't crash the simulator
- Model beats or matches the heuristic baseline

**Next phase (separate notebook):**
- Load SFT adapter as GRPO initialization
- Train with GRPO to optimize beyond the SFT policy
- Full curriculum benchmark across all 10 stages